In [3]:
from pyspark.sql.functions import *

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 4, Finished, Available, Finished, False)

In [4]:
bank_df = spark.read.table("bronze_bank_marketing")

customer_df = spark.read.table("bronze_customer")

channel_df = spark.read.table("bronze_channel")

region_df = spark.read.table("bronze_region")

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 5, Finished, Available, Finished, False)

In [5]:
bank_df = bank_df.toDF(*[
    c.replace(".", "_")
     .replace(" ", "_")
     .replace("-", "_")
     .lower()
    for c in bank_df.columns
])

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 6, Finished, Available, Finished, False)

In [6]:
bank_df = bank_df.dropna()

customer_df = customer_df.dropna()

channel_df = channel_df.dropna()

region_df = region_df.dropna()

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 7, Finished, Available, Finished, False)

In [7]:
print("Before :", bank_df.count())

print("Distinct :", bank_df.distinct().count())

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 8, Finished, Available, Finished, False)

Before : 19991
Distinct : 19991


In [8]:
bank_df = bank_df.dropDuplicates()

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 9, Finished, Available, Finished, False)

In [9]:
print("Before :", customer_df.count())

print("Distinct :", customer_df.distinct().count())

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 10, Finished, Available, Finished, False)

Before : 19991
Distinct : 19991


In [10]:
customer_df = customer_df.dropDuplicates()

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 11, Finished, Available, Finished, False)

In [11]:
print("Before :", channel_df.count())

print("Distinct :", channel_df.distinct().count())

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 12, Finished, Available, Finished, False)

Before : 8
Distinct : 8


In [12]:
channel_df = channel_df.dropDuplicates()

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 13, Finished, Available, Finished, False)

In [13]:
print("Before :", region_df.count())

print("Distinct :", region_df.distinct().count())

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 14, Finished, Available, Finished, False)

Before : 6
Distinct : 6


In [14]:
region_df = region_df.dropDuplicates()

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 15, Finished, Available, Finished, False)

In [15]:
bank_df = bank_df.replace("unknown", "Not Specified")

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 16, Finished, Available, Finished, False)

In [16]:
from pyspark.sql.functions import when, col
bank_df = bank_df.withColumn(
    "pdays",
    when(col("pdays") == 999, None)
    .otherwise(col("pdays"))
)
bank_df.select("pdays").show(10)

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 17, Finished, Available, Finished, False)

+-----+
|pdays|
+-----+
| NULL|
| NULL|
| NULL|
| NULL|
| NULL|
| NULL|
| NULL|
| NULL|
| NULL|
| NULL|
+-----+
only showing top 10 rows



In [17]:
from pyspark.sql.functions import to_date
bank_df = bank_df.withColumn(
    "account_open_date",
    to_date(col("account_open_date"))
)

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 18, Finished, Available, Finished, False)

In [18]:
bank_df.printSchema()

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 19, Finished, Available, Finished, False)

root
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- education: string (nullable = true)
 |-- default: string (nullable = true)
 |-- housing: string (nullable = true)
 |-- loan: string (nullable = true)
 |-- contact: string (nullable = true)
 |-- month: string (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- campaign: integer (nullable = true)
 |-- pdays: integer (nullable = true)
 |-- previous: integer (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- emp_var_rate: double (nullable = true)
 |-- cons_price_idx: double (nullable = true)
 |-- cons_conf_idx: double (nullable = true)
 |-- euribor3m: double (nullable = true)
 |-- nr_employed: double (nullable = true)
 |-- y: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- channel: string (nullabl

## Derived Columns

In [19]:
## 1. Customer Age Group
from pyspark.sql.functions import when

bank_df = bank_df.withColumn(
    "age_group",
    when(col("age").between(18,25), "18-25")
    .when(col("age").between(26,35), "26-35")
    .when(col("age").between(36,45), "36-45")
    .when(col("age").between(46,55), "46-55")
    .otherwise("55+")
)
display(bank_df)

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1a3580e2-c131-494c-9155-03cbe918f1be)

In [20]:
## 2. Income Band
bank_df = bank_df.withColumn(
    "income_band",
    when(col("monthly_income") < 25000, "Low")
    .when((col("monthly_income") >= 25000) & (col("monthly_income") < 50000), "Medium")
    .when((col("monthly_income") >= 50000) & (col("monthly_income") <= 100000), "High")
    .otherwise("Premium")
)

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 21, Finished, Available, Finished, False)

In [21]:
display(bank_df)

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 84d408a2-e040-41e5-817a-5d6c5667678b)

In [22]:
## 3. Contact Type Flag
bank_df = bank_df.withColumn(
    "contact_type_flag",
    when(col("contact") == "telephone", 0)
    .when(col("contact") == "cellular", 1)
    .otherwise(None)
)

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 23, Finished, Available, Finished, False)

In [23]:
display(bank_df)

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0325632b-2a1f-4078-b782-13fcfaf89a11)

In [24]:
## 4. Account Tenure
from pyspark.sql.functions import current_date, datediff, round, col

bank_df = bank_df.withColumn(
    "account_tenure_years",
    round(
        datediff(current_date(), col("account_open_date")) / 365,
        2
    )
)
display(bank_df)

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 170d2d24-1129-4c05-bc92-15a0938966d8)

In [25]:
from pyspark.sql.functions import col, lit

campaign_weight = 2

bank_df = bank_df.withColumn(
    "engagement_score",
    (col("duration") * lit(campaign_weight)) +
    col("previous")
)

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 26, Finished, Available, Finished, False)

In [26]:
bank_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_bank_marketing")

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 27, Finished, Available, Finished, False)

In [27]:
customer_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_customer")

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 28, Finished, Available, Finished, False)

In [28]:
channel_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_channel")

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 29, Finished, Available, Finished, False)

In [29]:
region_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_region")

StatementMeta(, c39f4509-4e93-4569-a417-bc522d544f71, 30, Finished, Available, Finished, False)